# Fashion-MNIST: Linear Softmax Classifier

This note studies **multinomial logistic regression**, implemented as one affine layer followed by a softmax function, with no hidden layers and no regularization. The goal is to establish a mathematically consistent baseline for subsequent GD and SGD analysis.

The derivations distinguish derivatives with respect to **logits** from derivatives with respect to the **trainable parameters**.

## 1. Notation and dimensions

Vectors are column vectors in the mathematics. Samples are stored as rows in the data and logit matrices. Lowercase letters denote scalars or vectors, and uppercase letters denote matrices; indices make the distinction explicit.

| Symbol | Meaning | Dimension or range |
| --- | --- | --- |
| $n$ | Number of training samples | $60{,}000$ |
| $d$ | Number of features per image | $28\times28=784$ |
| $K$ | Number of classes | $10$ |
| $i$ | Sample index | $1,\ldots,n$ |
| $r,s$ | Feature indices | $1,\ldots,d$ |
| $j,\ell$ | Class indices | $1,\ldots,K$ |
| $T$ | Raw training images | $\{0,\ldots,255\}^{n\times28\times28}$ |
| $x_i$ | Flattened and normalized image | $[0,1]^d$ |
| $X$ | Matrix with row $i$ equal to $x_i^\top$ | $[0,1]^{n\times d}$ |
| $c_i$ | Class label in mathematical notation | $\{1,\ldots,K\}$ |
| $y_i$ | One-hot target, with $y_{i,j}=\mathbf{1}_{\{j=c_i\}}$ | $\{0,1\}^K$, $\mathbf{1}_K^\top y_i=1$ |
| $Y$ | Matrix with row $i$ equal to $y_i^\top$ | $\{0,1\}^{n\times K}$ |
| $W$ | Trainable weight matrix | $\mathbb{R}^{d\times K}$ |
| $b$ | Trainable bias vector | $\mathbb{R}^K$ |
| $z_i$ | Logit vector for sample $i$ | $\mathbb{R}^K$ |
| $Z$ | Matrix with row $i$ equal to $z_i^\top$ | $\mathbb{R}^{n\times K}$ |
| $p_i$ | Predicted class probabilities for sample $i$ | $(0,1)^K$, $\mathbf{1}_K^\top p_i=1$ |
| $P$ | Matrix with row $i$ equal to $p_i^\top$ | $(0,1)^{n\times K}$ |
| $\mathbf{1}_q$ | Vector of ones | $\mathbb{R}^q$ |

The mathematical class labels are one-based; the Python labels are zero-based, from `0` to `K - 1`. All norms of vectors below are Euclidean. Matrix parameter gradients use the Frobenius inner product, $\langle A,B\rangle_F=\operatorname{tr}(A^\top B)$.

## 2. Load and preprocess Fashion-MNIST

Fashion-MNIST supplies separate training and test sets, with $60{,}000$ and $10{,}000$ images, respectively. The objective below uses only the training set. The test set is reserved for evaluation.

For each image, flatten its $28\times28$ pixels and divide the raw integer intensities by $255$:

$$
x_i=\frac{\operatorname{flatten}(T_i)}{255}\in[0,1]^d.
$$

The code reads `dataset.data` directly. Dataset transforms such as `ToTensor()` are applied when retrieving samples through `dataset[i]`, not when directly accessing `.data`; therefore, the conversion and normalization are explicit here.

**Notebook usage:** Select a Python 3 kernel with `torch` and `torchvision` installed, then run the code cells in order. The first data-loading cell downloads Fashion-MNIST if needed. Code cells are intentionally saved without execution counts or outputs so they can be run in your own environment.


In [1]:
import torch
import torch.nn.functional as F

torch.manual_seed(42)


In [2]:
def load_fashion_mnist(root="data", download=False):
    """Read raw uint8 tensors and normalize once; transforms are unnecessary."""
    from torchvision import datasets

    training_data = datasets.FashionMNIST(
        root=root, train=True, download=download
    )
    test_data = datasets.FashionMNIST(
        root=root, train=False, download=download
    )
    X = training_data.data.flatten(start_dim=1).to(torch.float32) / 255.0
    labels = training_data.targets.to(torch.long)
    Y = F.one_hot(labels, num_classes=10).to(X.dtype)
    X_test = test_data.data.flatten(start_dim=1).to(torch.float32) / 255.0
    test_labels = test_data.targets.to(torch.long)
    return X, Y, labels, X_test, test_labels


In [3]:
# Set download=True for the first run if the dataset is not already present.
X, Y, labels, X_test, test_labels = load_fashion_mnist(
    root="data", download=True
)
n, d = X.shape
K = Y.shape[1]

assert X.shape == (60000, 784)
assert Y.shape == (60000, 10)
assert X_test.shape == (10000, 784)
assert torch.all(Y.sum(dim=1) == 1)


The one-hot matrix is convenient for the derivation. An implementation can instead keep integer labels to avoid storing it, but this note retains the original one-hot formulation.

## 3. Affine model and row-wise softmax

For fixed parameters $(W,b)$, define

$$
z_i=W^\top x_i+b,
\qquad
Z=XW+\mathbf{1}_n b^\top.
$$

In Python, `X @ W + b` broadcasts `b` over samples. The stored shape of `b` remains `(K,)`; broadcasting does not modify the tensor's shape.

The draft's initialization is retained: independently sample the entries of $W$ and $b$ uniformly from $[0,1)$. This initialization is a modeling choice, not an assumption required for the derivative formulas or convexity.


In [4]:
def initialize_parameters(d, K, *, dtype=torch.float32, device="cuda"):
    """Retain the draft's U[0, 1) initialization; seed the caller for repeats."""
    W = torch.rand(d, K, dtype=dtype, device=device)
    b = torch.rand(K, dtype=dtype, device=device)
    return W, b


In [5]:
W, b = initialize_parameters(d, K, dtype=X.dtype, device=X.device)
Z = X @ W + b
assert Z.shape == (n, K)


For one sample, define $\sigma:\mathbb{R}^K\to(0,1)^K$ by

$$
\sigma(z_i)_j=p_{i,j}
=\frac{\exp(z_{i,j})}{\sum_{\ell=1}^K\exp(z_{i,\ell})}.
$$

The corresponding matrix map $\Sigma:\mathbb{R}^{n\times K}\to(0,1)^{n\times K}$ applies $\sigma$ independently to each row, so $P=\Sigma(Z)$. Each row of $P$ sums to one.

**Flattening the dataset does not turn this into one softmax over $nK$ entries.** Every normalization denominator still includes only the $K$ classes of its own sample.

For numerical stability, introduce a separate shifted quantity

$$
m_i=\max_j z_{i,j},\qquad
\widetilde z_i=z_i-m_i\mathbf{1}_K.
$$

Softmax is invariant to this shift. The mathematical model continues to use the original logits $Z=XW+\mathbf{1}_n b^\top$; shifting is an evaluation technique.

## 4. Per-sample loss and scalar objective

With the target $y_i$ fixed, the per-sample negative log-likelihood is

$$
\ell_i:\mathbb{R}^K\to\mathbb{R},\qquad
\ell_i(z_i)
=-\sum_{j=1}^K y_{i,j}\log p_{i,j}
=-\log p_{i,c_i}.
$$

Using $\sum_j y_{i,j}=1$ gives the equivalent expression

$$ 
\begin{aligned}
\ell_i(z_i) &=-\sum_{j=1}^K y_{i,j}\log p_{i,j}\\
&=-\sum_{j=1}^K y_{i,j}\log\left(\frac{\exp(z_{i,j})}{\sum_{\ell=1}^K\exp(z_{i,\ell})}\right)\\
&=-\left(\sum_{j=1}^K y_{i,j} z_{i,j} - \underbrace{\sum_{j=1}^K y_{i,j}}_{=1} \log\sum_{j=1}^K\exp(z_{i,j}) \right)
\end{aligned}
$$

$$
\boxed{
\ell_i(z_i)=\log\left(\sum_{j=1}^K e^{z_{i,j}}\right)-y_i^\top z_i.
}
$$

For finite logits and $K\ge2$, $0<p_{i,c_i}<1$, so $\ell_i(z_i)>0$ in exact arithmetic. In particular, zero is a lower bound. Floating-point evaluation may round a sufficiently small loss to zero.

Distinguish three objects:

$$
\mathcal{L}(Z)=\begin{bmatrix}\ell_1(z_1)\\\vdots\\\ell_n(z_n)\end{bmatrix}
\in\mathbb{R}^n,
\qquad
\Phi(Z)=\frac1n\sum_{i=1}^n\ell_i(z_i),
$$

$$
\boxed{
f(W,b)=\Phi(XW+\mathbf{1}_n b^\top)
=\frac1n\sum_{i=1}^n f_i(W,b),
\qquad
f_i(W,b)=\ell_i(W^\top x_i+b).
}
$$

$\mathcal{L}$ is vector-valued, whereas $\Phi$ and $f$ are scalar-valued. Training minimizes $f$ over $(W,b)$; the logits are determined by the fixed data and these shared parameters.

### Stable implementation

Computing `torch.log(P)` after forming probabilities can produce `-inf` when a probability underflows to zero. Multiplying this by a zero one-hot entry can then produce `NaN`. Compute log-probabilities directly instead:

$$
\log p_{i,j}=\widetilde z_{i,j}
-\log\left(\sum_{\ell=1}^K e^{\widetilde z_{i,\ell}}\right).
$$


In [6]:
def log_softmax_rows(Z):
    """Stable row-wise log probabilities without taking log(softmax(Z))."""
    shifted = Z - Z.amax(dim=1, keepdim=True)
    return shifted - torch.logsumexp(shifted, dim=1, keepdim=True)


def forward(X, Y, W, b):
    """Y must contain one-hot rows (or valid probability distributions)."""
    Z = X @ W + b
    log_P = log_softmax_rows(Z)
    P = log_P.exp()
    losses = -(Y * log_P).sum(dim=1)
    return P, losses


def objective(X, Y, W, b):
    return forward(X, Y, W, b)[1].mean()


In [7]:
P, losses = forward(X, Y, W, b)
f_value = losses.mean()

assert losses.shape == (n,)
assert f_value.ndim == 0
torch.testing.assert_close(P.sum(dim=1), torch.ones_like(P[:, 0]))
torch.testing.assert_close(f_value, F.cross_entropy(X @ W + b, labels))


## 5. First-order derivatives

### 5.1 Softmax Jacobian for one sample

Temporarily omit the sample index and write $p=\sigma(z)$. For $j,\ell\in\{1,\ldots,K\}$,

$$
\frac{\partial p_j}{\partial z_\ell}
=\begin{cases}
p_j(1-p_j),&j=\ell,\\
-p_jp_\ell,&j\ne\ell.
\end{cases}
=p_j(\delta_{j\ell}-p_\ell).
$$

Thus,

$$
\boxed{J_\sigma(z)=\operatorname{Diag}(p)-pp^\top\in\mathbb{R}^{K\times K}.}
$$

Here $\delta_{j\ell}$ is the Kronecker delta.  

$$
\delta_{j\ell}
=\begin{cases}
1, j = \ell \\
0, j \neq \ell
\end{cases}
$$


Use **Jacobian** notation $J_\sigma$ (or $D\sigma$) for the derivative of this vector-valued function. Its entries are not all positive: the off-diagonal entries are negative for finite logits.

### 5.2 Loss gradient with respect to logits

For one sample, the chain rule gives

$$
\begin{aligned}
\frac{\partial\ell_i}{\partial z_{i,\ell}}
&=-\sum_{j=1}^K y_{i,j}\frac{1}{p_{i,j}}
\frac{\partial p_{i,j}}{\partial z_{i,\ell}}\\
&=-\sum_{j=1}^K y_{i,j}(\delta_{j\ell}-p_{i,\ell})\\
&=-\sum_{j=1}^K y_{i,j}\delta_{j\ell}+\sum_{j=1}^K y_{i,j}p_{i,\ell}\\
&=-y_{i,\ell}+p_{i,\ell}\underbrace{\sum_{j=1}^K y_{i,j}}_{=1}\\
&=p_{i,\ell}-y_{i,\ell}.
\end{aligned}
$$

Therefore,

$$
\boxed{\nabla_{z_i}\ell_i(z_i)=p_i-y_i\in\mathbb{R}^K,}
\qquad
\boxed{\nabla_Z\Phi(Z)=\frac1n(P-Y)\in\mathbb{R}^{n\times K}.}
$$

These gradients can have negative components even though the losses are nonnegative. A nonnegative loss does not imply a nonnegative gradient.

### 5.3 Gradients with respect to weights and bias

Since $z_{i,j}=\sum_{r=1}^d x_{i,r}w_{r,j}+b_j$,

$$
\frac{\partial f}{\partial w_{r,j}}
=\frac1n\sum_{i=1}^n x_{i,r}(p_{i,j}-y_{i,j}),
\qquad
\frac{\partial f}{\partial b_j}
=\frac1n\sum_{i=1}^n(p_{i,j}-y_{i,j}).
$$

In matrix form,

$$
\boxed{\nabla_W f(W,b)=\frac1n X^\top(P-Y)\in\mathbb{R}^{d\times K},}
$$

$$
\boxed{\nabla_b f(W,b)=\frac1n(P-Y)^\top\mathbf{1}_n\in\mathbb{R}^{K}.}
$$


In [8]:
def manual_gradient(X, Y, W, b):
    """Gradient of the MEAN loss; shapes are (d, K) and (K,)."""
    n = X.shape[0]
    P, _ = forward(X, Y, W, b)
    residual = P - Y
    grad_W = X.T @ residual / n
    grad_b = residual.mean(dim=0)
    return grad_W, grad_b


In [9]:
grad_W, grad_b = manual_gradient(X, Y, W, b)
assert grad_W.shape == W.shape  # (784, 10)
assert grad_b.shape == b.shape  # (10,)


The original expression `(X.T @ (P - Y)).mean(dim=0)` is incorrect: matrix multiplication has already summed over samples. Applying `.mean(dim=0)` afterward averages over the **feature** dimension, removes that dimension, and returns a vector of length $K$. Divide the matrix product by $n$ instead.

# 6. Second-order derivatives

## Second-order derivatives with respect logits (z)

From the first-order derivate $ \frac{\partial\ell_i}{\partial z_{i,\ell}} = p_{i,\ell}-y_{i,\ell} $ we can get its second-order derivative:
$$
\begin{aligned}
\frac{\partial^2 \ell_i}{\partial z_{i,j}^2} = \frac{\partial^2 \ell_i}{\partial z_{i,\ell}^2} &= \frac{\partial \left(p_{i,\ell}-y_{i,\ell}\right)}{\partial z_{i,\ell}} \\
&= p_{i, \ell}(1-p_{i, \ell})
\end{aligned}
$$

also included

$$
\begin{aligned}
\frac{\partial^2 \ell_i}{\partial z_{i,j} z_{i,\ell}} = \frac{\partial^2 \ell_i}{\partial z_{i,\ell} z_{i,j}} &= \frac{\partial \left(p_{i,\ell}-y_{i,\ell}\right)}{\partial z_{i,j}} \\
&= p_{i,j}(\delta_{i,j\ell}-p_{i,\ell})
\end{aligned}
$$

Therefore, define $ C_i $

$$
\boxed{C_i := \nabla_{z}^2\ell_i(z_i)=\operatorname{Diag}(p_i)-p_ip_i^\top}
$$

# 7. Checking convexity

## Definition of Convexity
Let $ f : \mathbb{R}^d \to \mathbb{R} \cup \{+\infty\} $ is convex if for all $ x, y \in \mathbb{R}^d $ for all $ t \in (0,1),\qquad f(tx+(1-t)y)\leq tf(x) + (1-t)f(y) $

## First-order characterization
If $ f : \mathbb{R}^d \to \mathbb{R} $ is convex and differentiable then, for all $ x, y \in \mathbb{R}^d,\qquad f(x) \geq f(y) + \langle \nabla f(y), x - y \rangle $

## Second-order characterization
Let $ f : \mathbb{R}^d \to \mathbb{R} $ be convex and twice differentiable.  Then, for all $ x \in \mathbb{R}^d $, for every eigenvalue $ \lambda $ of $ \nabla^2 f(x) $, we have $ \lambda \geq 0 $ (Positive semi-definite)

## Proof in our objective function
To prove this objective function, we must ensure the objective function satisfies with definition of convexity.  We can use second-order characterization to check whether the function is convex.  

Based on definition of convexity, we can define $ y = x + tv $ where $ x $ dan $ y $ is the first and second point, then $ t \in (0,1) $ the factor of any vectors $ v $.  Therefore $ z_i(t) = z_i + tv $ where $ z_i $ is the logits.

Continue to find its derivative, suppose $ h(t) $ is a loss function where $ h(t) = \ell_i(z_i(t)) $, then we have :
$$
\begin{aligned}
h'(t) &= \underbrace{\nabla_z \ell_i(z_i(t))^\top}_{\in \mathbb{R}^{1 \times K}} \underbrace{\frac{d(z_i)}{dt}}_{\in \mathbb{R}^{K}} \\
&= \left(p_i(t) - y_i\right)^\top u
\end{aligned}
$$
for first-order derivation, and  
$$
\begin{aligned}
h''(t) = \frac{dh'(t)}{dt} &= \frac{d\left(p_i(t) - y_i \right) u}{dt} \\
&= \underbrace{u^\top}_{\in \mathbb{R}^{1 \times K}} \underbrace{\nabla_{z}^2 \ell_i\left(z_i(t)\right)}_{\in \mathbb{R}^{K \times K}}\underbrace{u}_{\in \mathbb{R}^K} \\
&= u^\top C_i u
\end{aligned}
$$
for second-order derivation  

We can expand the equation  
$$
\begin{aligned}
u^\top C_i u &= u^\top \left(\operatorname{Diag}(p_i)-p_ip_i^\top\right) u \\
&= u^\top\operatorname{Diag}(p_i)u - u^\top p_i p_i^\top u \\
&= \sum_{j=1}^K p_{i,j} u_j^2 - \left(\sum_{j=1}^K p_{i,j} u_j\right)^2 \\
\end{aligned}
$$

Suppose $ \bar u = \sum_{j=1}^K p_{i,j} u_j $ then
$$
\begin{aligned}
\sum_{j=1}^K p_{i,j} u_j^2 - \left(\sum_{j=1}^K p_{i,j} u_j\right)^2 &= \sum_{j=1}^K p_{i,j} u_j^2 - \bar u^2 \\
&= \sum_{j=1}^K p_{i,j} u_j^2 + \bar u^2 - 2 \bar u^2 \\
&= \sum_{j=1}^K p_{i,j} u_j^2 + \bar u^2 \sum_{j=1}^K p_{i,j} - 2 \bar u \left(\sum_{j=1}^K p_{i,j} u_j\right)
\end{aligned}
$$


# 8. Convexity with respect to $W$ and $b$

Section 7 studies one loss $\ell_i$ as a function of its logits. Training, however, minimizes $f(W,b)$, so convexity must be established in the **parameters**. The argument has two ingredients: (i) $C_i\succeq0$ for every logit vector, and (ii) the map $(W,b)\mapsto z_i$ is affine.

## 8.1 Completing the logit-level argument

Continue the expansion at the end of Section 7. With $\bar u=\sum_{j}p_{i,j}u_j$ and $\sum_j p_{i,j}=1$, the last line is a sum of squares:

$$
\begin{aligned}
u^\top C_iu
&=\sum_{j=1}^K p_{i,j}u_j^2-2\bar u\sum_{j=1}^K p_{i,j}u_j+\bar u^2\sum_{j=1}^K p_{i,j}\\
&=\sum_{j=1}^K p_{i,j}\left(u_j^2-2\bar u\,u_j+\bar u^2\right)\\
&=\sum_{j=1}^K p_{i,j}(u_j-\bar u)^2\;\ge\;0.
\end{aligned}
$$

This is the variance of the entries of $u$ under the distribution $p_i$. Hence $C_i\succeq0$ for every $z_i\in\mathbb{R}^K$, so $h''(t)\ge0$ along every line, and each $\ell_i$ is convex in $z_i$.

The inequality is not strict: $C_i\mathbf{1}_K=p_i-p_i(\mathbf{1}_K^\top p_i)=0$. Along $u=\mathbf{1}_K$, every logit increases by the same amount, the softmax output does not change, and the curvature is zero.

## 8.2 The parameter-to-logit map is affine

Write $\theta=(W,b)\in\mathbb{R}^{d\times K}\times\mathbb{R}^K$ and use

$$
\langle\theta,\theta'\rangle=\langle W,W'\rangle_F+b^\top b',\qquad \|\theta\|^2=\|W\|_F^2+\|b\|^2.
$$

Fix a point $\theta$ and an arbitrary direction $\delta=(V,c)$ with $V\in\mathbb{R}^{d\times K}$ and $c\in\mathbb{R}^K$. Along the line $\theta+t\delta$,

$$
z_i(t)=(W+tV)^\top x_i+(b+tc)=z_i+t\,u_i,\qquad \boxed{u_i:=V^\top x_i+c\in\mathbb{R}^K.}
$$

The logit direction $u_i$ depends on the sample and on $\delta$, but **not on $t$**: a line in parameter space is mapped to a line in each sample's logit space. This is exactly the situation of Section 7, now with a different direction $u_i$ for every sample.

## 8.3 Second derivative along an arbitrary line

Define $g(t)=f(\theta+t\delta)=\frac1n\sum_{i=1}^n\ell_i(z_i+tu_i)$. Applying Section 7 to each term,

$$
g'(t)=\frac1n\sum_{i=1}^n\big(p_i(t)-y_i\big)^\top u_i,
\qquad
\boxed{g''(t)=\frac1n\sum_{i=1}^n u_i^\top C_i(t)\,u_i\;\ge\;0,}
$$

where $p_i(t)$ and $C_i(t)$ are evaluated at $z_i(t)$. Each summand is nonnegative by 8.1.

A function is convex if and only if its restriction to every line is convex, and a twice-differentiable function of one variable with $g''\ge0$ is convex. Since $\theta$ and $\delta$ were arbitrary,

$$
\boxed{f(W,b)\ \text{is convex in}\ (W,b).}
$$

Consistency with Section 5.3: since $(p_i-y_i)^\top u_i=x_i^\top V(p_i-y_i)+(p_i-y_i)^\top c$,

$$
g'(0)=\Big\langle \tfrac1nX^\top(P-Y),\,V\Big\rangle_F+\Big(\tfrac1n(P-Y)^\top\mathbf 1_n\Big)^\top c
=\langle\nabla f(\theta),\delta\rangle,
$$

which recovers $\nabla_Wf$ and $\nabla_bf$.

## 8.4 The Hessian in matrix form

Absorb the bias into an augmented weight matrix:

$$
\widetilde x_i=\begin{bmatrix}x_i\\1\end{bmatrix}\in\mathbb{R}^{d+1},\qquad
\widetilde W=\begin{bmatrix}W\\b^\top\end{bmatrix}\in\mathbb{R}^{(d+1)\times K},\qquad
\widetilde X=\begin{bmatrix}X&\mathbf 1_n\end{bmatrix}\in\mathbb{R}^{n\times(d+1)}.
$$

Then $z_i=\widetilde W^\top\widetilde x_i$ and $Z=\widetilde X\widetilde W$. With $\widetilde V=\begin{bmatrix}V\\c^\top\end{bmatrix}$, the logit direction is $u_i=\widetilde V^\top\widetilde x_i=(I_K\otimes\widetilde x_i^\top)\operatorname{vec}(\widetilde V)$, where $\operatorname{vec}$ stacks columns. Therefore $g''(0)=\operatorname{vec}(\widetilde V)^\top H\operatorname{vec}(\widetilde V)$ with

$$
\boxed{H=\nabla^2 f(W,b)=\frac1n\sum_{i=1}^n C_i\otimes\widetilde x_i\widetilde x_i^\top\in\mathbb{R}^{(d+1)K\times(d+1)K}.}
$$

The eigenvalues of a Kronecker product are the products of the eigenvalues of its factors. Both $C_i$ and $\widetilde x_i\widetilde x_i^\top$ are positive semidefinite, so each term, and therefore $H$, is positive semidefinite. For Fashion-MNIST, $H$ is $7850\times7850$.

## 8.5 Convex, but not strictly convex

Choose $\delta=(V,c)=(a\mathbf 1_K^\top,\ \alpha\mathbf 1_K)$ for any $a\in\mathbb{R}^d$ and $\alpha\in\mathbb{R}$. Then $u_i=(a^\top x_i+\alpha)\mathbf 1_K$, every logit of sample $i$ shifts by the same amount, and

$$
\ell_i(z_i+s\mathbf 1_K)=\log\sum_{j=1}^Ke^{z_{i,j}+s}-y_i^\top z_i-s=\ell_i(z_i).
$$

So $f$ is constant along these directions and $g''\equiv0$. Consequently:

- $H$ is singular at every point, so $f$ is **not strongly convex**, and minimizers, when they exist, are not unique.
- The GD analysis in Section 10 relies only on convexity and $L$-smoothness, and states its guarantee against an arbitrary reference point instead of a unique minimizer.

The next cells verify these statements numerically. The checks run in float64, so rounding does not hide the inequalities.

In [ ]:
X_dbl, Y_dbl = X.double(), Y.double()
W_dbl, b_dbl = W.double(), b.double()


def parameter_norm(W, b):
    """||theta|| = sqrt(||W||_F^2 + ||b||^2)."""
    return (W.square().sum() + b.square().sum()).sqrt()


def directional_curvature(X, Y, W, b, V, c):
    """g''(0) = (1/n) sum_i u_i^T C_i u_i, evaluated as a variance under p_i."""
    P, _ = forward(X, Y, W, b)
    U = X @ V + c
    U_bar = (P * U).sum(dim=1, keepdim=True)
    return (P * (U - U_bar).square()).sum(dim=1).mean()


def autograd_curvature(X, Y, W, b, V, c):
    """The same <delta, H delta>, from an autograd Hessian-vector product."""
    _, (HV, Hc) = torch.autograd.functional.hvp(
        lambda W_, b_: objective(X, Y, W_, b_), (W, b), (V, c)
    )
    return (V * HV).sum() + (c * Hc).sum()

In [ ]:
generator = torch.Generator().manual_seed(0)

# Section 8.3: g''(0) >= 0 along random directions, and it matches autograd.
for _ in range(5):
    V = torch.randn(d, K, dtype=torch.float64, generator=generator)
    c = torch.randn(K, dtype=torch.float64, generator=generator)
    curvature = directional_curvature(X_dbl, Y_dbl, W_dbl, b_dbl, V, c)
    torch.testing.assert_close(
        curvature, autograd_curvature(X_dbl, Y_dbl, W_dbl, b_dbl, V, c)
    )
    assert curvature >= 0

# Section 8.5: zero curvature along (V, c) = (a 1_K^T, alpha 1_K).
a = torch.randn(d, 1, dtype=torch.float64, generator=generator)
V_flat = a * torch.ones(1, K, dtype=torch.float64)
c_flat = 2.0 * torch.ones(K, dtype=torch.float64)
assert directional_curvature(X_dbl, Y_dbl, W_dbl, b_dbl, V_flat, c_flat).abs() < 1e-12
torch.testing.assert_close(
    objective(X_dbl, Y_dbl, W_dbl + V_flat, b_dbl + c_flat),
    objective(X_dbl, Y_dbl, W_dbl, b_dbl),
)

# Definition of convexity on a chord between two parameter points.
W2 = torch.randn(d, K, dtype=torch.float64, generator=generator)
b2 = torch.randn(K, dtype=torch.float64, generator=generator)
f1 = objective(X_dbl, Y_dbl, W_dbl, b_dbl)
f2 = objective(X_dbl, Y_dbl, W2, b2)
for t in torch.linspace(0.1, 0.9, 9, dtype=torch.float64):
    f_t = objective(X_dbl, Y_dbl, t * W_dbl + (1 - t) * W2, t * b_dbl + (1 - t) * b2)
    assert f_t <= t * f1 + (1 - t) * f2 + 1e-12

# 9. $L$-smoothness

## 9.1 Definition and two facts used later

A differentiable $f$ is **$L$-smooth** if its gradient is $L$-Lipschitz:

$$
\|\nabla f(\theta)-\nabla f(\theta')\|\le L\|\theta-\theta'\|\qquad\text{for all }\theta,\theta'.
$$

**Fact 1 (Hessian criterion).** If $f$ is twice differentiable and $\|\nabla^2f(\theta)\|_2\le L$ for all $\theta$, then $f$ is $L$-smooth. For convex $f$, this condition is $0\preceq\nabla^2f(\theta)\preceq LI$, that is, $0\le\delta^\top\nabla^2f(\theta)\,\delta\le L\|\delta\|^2$ for every direction $\delta$.

*Proof.* Apply the fundamental theorem of calculus to $s\mapsto\nabla f(\theta'+s(\theta-\theta'))$:

$$
\nabla f(\theta)-\nabla f(\theta')=\int_0^1\nabla^2f\big(\theta'+s(\theta-\theta')\big)(\theta-\theta')\,ds,
$$

so $\|\nabla f(\theta)-\nabla f(\theta')\|\le\int_0^1 L\|\theta-\theta'\|\,ds=L\|\theta-\theta'\|$. For a symmetric positive semidefinite matrix, the spectral norm is the largest eigenvalue, which is the maximum of $\delta^\top H\delta/\|\delta\|^2$. $\square$

**Fact 2 (descent lemma).** If $f$ is $L$-smooth, then for all $\theta,\theta'$,

$$
f(\theta')\le f(\theta)+\langle\nabla f(\theta),\theta'-\theta\rangle+\frac L2\|\theta'-\theta\|^2.
$$

*Proof.* Let $\Delta=\theta'-\theta$. By Cauchy–Schwarz and $L$-smoothness,

$$
f(\theta')-f(\theta)-\langle\nabla f(\theta),\Delta\rangle=\int_0^1\big\langle\nabla f(\theta+s\Delta)-\nabla f(\theta),\Delta\big\rangle\,ds
\le\int_0^1 Ls\|\Delta\|^2\,ds=\frac L2\|\Delta\|^2.\qquad\square
$$

## 9.2 Smoothness in the logits: $0\preceq C_i\preceq\tfrac12 I_K$

Bound the eigenvalues of $C_i=\operatorname{Diag}(p_i)-p_ip_i^\top$ with the **Gershgorin circle theorem**: every eigenvalue lies in a disc centered at some diagonal entry, with radius equal to the sum of the absolute off-diagonal entries in that row. In row $j$,

$$
\text{center}=p_{i,j}(1-p_{i,j}),\qquad
\text{radius}=\sum_{\ell\ne j}p_{i,j}p_{i,\ell}=p_{i,j}(1-p_{i,j}).
$$

Therefore every eigenvalue $\lambda$ of $C_i$ satisfies, for some $j$,

$$
\lambda\le 2p_{i,j}(1-p_{i,j})\le 2\cdot\frac14=\frac12,
$$

because $p(1-p)\le\frac14$ for $p\in[0,1]$. Combined with 8.1,

$$
\boxed{0\preceq C_i\preceq\tfrac12 I_K\quad\text{for every }z_i\in\mathbb{R}^K.}
$$

The constant is attained: for $K=2$ and $p_i=(\tfrac12,\tfrac12)$, $C_i=\tfrac14\begin{bmatrix}1&-1\\-1&1\end{bmatrix}$ has eigenvalue $\tfrac12$.

By Fact 1, each $\ell_i$ is $\tfrac12$-smooth in its logits; equivalently, softmax is $\tfrac12$-Lipschitz:

$$
\|\nabla_{z}\ell_i(z_i)-\nabla_{z}\ell_i(z_i')\|=\|\sigma(z_i)-\sigma(z_i')\|\le\tfrac12\|z_i-z_i'\|.
$$

Since $\ell_i$ depends only on row $i$ of $Z$, the Hessian of $\Phi$ is block diagonal with blocks $\frac1nC_i$. Hence

$$
\boxed{\|\nabla_Z\Phi(Z)-\nabla_Z\Phi(Z')\|_F=\tfrac1n\|P-P'\|_F\le\tfrac1{2n}\|Z-Z'\|_F,\qquad L_\Phi=\tfrac1{2n}.}
$$

## 9.3 Transfer from the logits to $W$ and $b$

Use the line from Section 8: direction $\delta=(V,c)$ and logit directions $u_i=\widetilde V^\top\widetilde x_i$. By 9.2,

$$
\begin{aligned}
\delta^\top\nabla^2f(\theta)\,\delta=g''(0)
&=\frac1n\sum_{i=1}^n u_i^\top C_iu_i
\le\frac1{2n}\sum_{i=1}^n\|u_i\|^2
=\frac1{2n}\|\widetilde X\widetilde V\|_F^2\\
&\le\frac{\|\widetilde X\|_2^2}{2n}\,\|\widetilde V\|_F^2
=\frac{\lambda_{\max}(\widetilde X^\top\widetilde X)}{2n}\,\|\delta\|^2.
\end{aligned}
$$

The equalities use that row $i$ of $\widetilde X\widetilde V$ is $u_i^\top$, and $\|\widetilde V\|_F^2=\|V\|_F^2+\|c\|^2=\|\delta\|^2$. The second inequality applies $\|\widetilde Xv\|\le\|\widetilde X\|_2\|v\|$ to each column $v$ of $\widetilde V$. Together with $g''\ge0$ from Section 8, $0\preceq\nabla^2f\preceq LI$, and Fact 1 gives

$$
\boxed{f\ \text{is}\ L\text{-smooth in}\ (W,b),\qquad L=\frac{\lambda_{\max}(\widetilde X^\top\widetilde X)}{2n}=\frac{\sigma_{\max}(\widetilde X)^2}{2n}.}
$$

In chain-rule terms, $f(\theta)=\Phi(\widetilde X\widetilde W)$ composes the $\frac1{2n}$-smooth $\Phi$ with a linear map of operator norm $\|\widetilde X\|_2$, and a linear map multiplies the smoothness constant by its squared norm.

**Cruder, data-independent bounds.** Since $\lambda_{\max}\le\operatorname{tr}$ for positive semidefinite matrices and $x_i\in[0,1]^d$,

$$
L\le\frac1{2n}\operatorname{tr}(\widetilde X^\top\widetilde X)=\frac1{2n}\sum_{i=1}^n\big(\|x_i\|^2+1\big)\le\frac{d+1}2=392.5.
$$

The cells below compute the data-dependent $L$, which Section 10 uses as the step-size scale, and check the bounds numerically.

In [ ]:
def smoothness_constant(X):
    """L = lambda_max(X_aug^T X_aug) / (2n), with X_aug = [X, 1_n] (Section 9.3)."""
    n = X.shape[0]
    ones = torch.ones(n, 1, dtype=X.dtype, device=X.device)
    X_aug = torch.cat([X, ones], dim=1).double()
    return torch.linalg.eigvalsh(X_aug.T @ X_aug)[-1].item() / (2 * n)


L = smoothness_constant(X)
L_trace = (X_dbl.square().sum().item() + n) / (2 * n)
print(f"L = {L:.4f}   trace bound = {L_trace:.4f}   (d + 1) / 2 = {(d + 1) / 2}")
assert L <= L_trace <= (d + 1) / 2

In [ ]:
# Section 9.2: eigenvalues of C_i lie in [0, 1/2] for arbitrary logits.
Z_random = 3.0 * torch.randn(2000, K, dtype=torch.float64, generator=generator)
P_random = torch.softmax(Z_random, dim=1)
C = torch.diag_embed(P_random) - P_random[:, :, None] * P_random[:, None, :]
eigenvalues = torch.linalg.eigvalsh(C)
assert eigenvalues.min() >= -1e-12 and eigenvalues.max() <= 0.5
print(f"largest eigenvalue of C_i over 2000 random logits: {eigenvalues.max():.4f}")

# Section 9.3: Rayleigh quotients of the parameter Hessian lie in [0, L].
for _ in range(5):
    V = torch.randn(d, K, dtype=torch.float64, generator=generator)
    c = torch.randn(K, dtype=torch.float64, generator=generator)
    rayleigh = directional_curvature(X_dbl, Y_dbl, W_dbl, b_dbl, V, c) / parameter_norm(V, c).square()
    assert 0 <= rayleigh <= L

# Definition of L-smoothness: ||grad f(theta) - grad f(theta')|| <= L ||theta - theta'||.
for scale in [1e-3, 1e-1, 1e1]:
    V = scale * torch.randn(d, K, dtype=torch.float64, generator=generator)
    c = scale * torch.randn(K, dtype=torch.float64, generator=generator)
    grad_W1, grad_b1 = manual_gradient(X_dbl, Y_dbl, W_dbl, b_dbl)
    grad_W2, grad_b2 = manual_gradient(X_dbl, Y_dbl, W_dbl + V, b_dbl + c)
    ratio = parameter_norm(grad_W1 - grad_W2, grad_b1 - grad_b2) / parameter_norm(V, c)
    assert ratio <= L
    print(f"step scale {scale:g}: gradient ratio = {ratio:.4f} <= L = {L:.4f}")

# 10. Gradient descent and its convergence

## 10.1 Algorithm

Starting from $\theta_0=(W_0,b_0)$, full-batch gradient descent with step size $\eta>0$ iterates

$$
\theta_{k+1}=\theta_k-\eta\nabla f(\theta_k).
$$

With $P_k=\Sigma(XW_k+\mathbf 1_nb_k^\top)$ and the gradients from Section 5.3, this is

$$
W_{k+1}=W_k-\frac\eta n X^\top(P_k-Y),\qquad
b_{k+1}=b_k-\frac\eta n(P_k-Y)^\top\mathbf 1_n.
$$

Write $g_k=\nabla f(\theta_k)$.

## 10.2 Theorem

Let $f$ be convex (Section 8) and $L$-smooth (Section 9), and let $0<\eta\le\frac1L$. Then for every $T\ge1$ and **every** reference point $\theta_{\mathrm{ref}}$:

**(a) Sufficient decrease.** $f(\theta_{k+1})\le f(\theta_k)-\frac\eta2\|g_k\|^2$ for every $k$; in particular, $f(\theta_k)$ is nonincreasing.

**(b) Objective gap.**
$$
f(\theta_T)-f(\theta_{\mathrm{ref}})\le\frac{\|\theta_0-\theta_{\mathrm{ref}}\|^2}{2\eta T}.
$$

**(c) Gradient norm.**
$$
\min_{0\le k<T}\|g_k\|^2\le\frac{2\big(f(\theta_0)-f(\theta_T)\big)}{\eta T}\le\frac{2f(\theta_0)}{\eta T}.
$$

With $\eta=\frac1L$ and $\theta_{\mathrm{ref}}=\theta^\star$ a minimizer, (b) is the classical rate

$$
\boxed{f(\theta_T)-f(\theta^\star)\le\frac{L\|\theta_0-\theta^\star\|^2}{2T}=\mathcal O\!\left(\frac1T\right).}
$$

## 10.3 Proof

**Step 1: (a) from the descent lemma.** Apply Fact 2 with $\theta=\theta_k$ and $\theta'=\theta_{k+1}=\theta_k-\eta g_k$:

$$
f(\theta_{k+1})\le f(\theta_k)-\eta\|g_k\|^2+\frac{L\eta^2}2\|g_k\|^2
=f(\theta_k)-\eta\Big(1-\frac{L\eta}2\Big)\|g_k\|^2
\le f(\theta_k)-\frac\eta2\|g_k\|^2,
$$

because $L\eta\le1$ implies $1-\frac{L\eta}2\ge\frac12$.

**Step 2: (c) by telescoping.** Summing (a) over $k=0,\ldots,T-1$,

$$
\frac\eta2\sum_{k=0}^{T-1}\|g_k\|^2\le f(\theta_0)-f(\theta_T)\le f(\theta_0),
$$

where the last step uses $f>0$ (Section 4). The minimum is at most the average, which gives (c). This step uses smoothness only, not convexity.

**Step 3: a one-step bound from convexity.** The first-order characterization (Section 7) at $\theta_k$ gives $f(\theta_k)\le f(\theta_{\mathrm{ref}})+\langle g_k,\theta_k-\theta_{\mathrm{ref}}\rangle$. Inserting this into (a),

$$
f(\theta_{k+1})\le f(\theta_{\mathrm{ref}})+\langle g_k,\theta_k-\theta_{\mathrm{ref}}\rangle-\frac\eta2\|g_k\|^2.
$$

Expanding the update,

$$
\|\theta_{k+1}-\theta_{\mathrm{ref}}\|^2=\|\theta_k-\theta_{\mathrm{ref}}\|^2-2\eta\langle g_k,\theta_k-\theta_{\mathrm{ref}}\rangle+\eta^2\|g_k\|^2,
$$

so the last two terms above equal $\frac1{2\eta}\big(\|\theta_k-\theta_{\mathrm{ref}}\|^2-\|\theta_{k+1}-\theta_{\mathrm{ref}}\|^2\big)$. Hence

$$
f(\theta_{k+1})-f(\theta_{\mathrm{ref}})\le\frac1{2\eta}\Big(\|\theta_k-\theta_{\mathrm{ref}}\|^2-\|\theta_{k+1}-\theta_{\mathrm{ref}}\|^2\Big).
$$

**Step 4: (b) by telescoping.** Summing over $k=0,\ldots,T-1$, the right-hand side telescopes:

$$
\sum_{k=1}^{T}\big(f(\theta_k)-f(\theta_{\mathrm{ref}})\big)\le\frac{\|\theta_0-\theta_{\mathrm{ref}}\|^2-\|\theta_T-\theta_{\mathrm{ref}}\|^2}{2\eta}\le\frac{\|\theta_0-\theta_{\mathrm{ref}}\|^2}{2\eta}.
$$

By (a), $f(\theta_T)\le f(\theta_k)$ for every $k\le T$, so the left-hand side is at least $T\big(f(\theta_T)-f(\theta_{\mathrm{ref}})\big)$. Dividing by $T$ proves (b). $\square$

## 10.4 Remarks

- **Why an arbitrary reference point.** Section 8.5 shows that minimizers are never unique, and for a linear softmax model a minimizer need not exist at all: if the classes are linearly separable, the loss approaches $0$ only as $\|\theta\|\to\infty$. The proof of (b) never uses optimality of $\theta_{\mathrm{ref}}$, so it still applies. For any $\varepsilon>0$, choose $\theta_{\mathrm{ref}}$ with $f(\theta_{\mathrm{ref}})\le\inf f+\varepsilon$; then $\limsup_{T\to\infty}f(\theta_T)\le\inf f+\varepsilon$, hence $f(\theta_T)\to\inf f$.
- **No linear rate is guaranteed.** Without strong convexity, the general guarantee is the sublinear $\mathcal O(1/T)$ rate. Reaching accuracy $\varepsilon$ takes at most $T=\big\lceil L\|\theta_0-\theta^\star\|^2/(2\varepsilon)\big\rceil$ iterations when a minimizer exists.
- **Step size.** $\eta=1/L$ comes from a worst-case curvature bound. Monotone decrease in Step 1 only needs $1-L\eta/2>0$, that is, $\eta<2/L$, with a smaller decrease constant.

## 10.5 Implementation and numerical verification

The code below runs full-batch GD with $\eta=1/L$ from the initialization of Section 3, in float64, and checks (a), (b) and (c) at every iteration. For (b), the reference point is the final iterate, $\theta_{\mathrm{ref}}=\theta_T$; this is valid because the theorem holds for every reference point.

In [ ]:
def gradient_descent(X, Y, W, b, step_size, num_steps):
    """Full-batch GD on f; returns theta_T, f(theta_k) for k <= T and ||g_k||^2 for k < T."""
    f_values, grad_sq_norms = [], []
    for _ in range(num_steps):
        f_values.append(objective(X, Y, W, b).item())
        grad_W, grad_b = manual_gradient(X, Y, W, b)
        grad_sq_norms.append(parameter_norm(grad_W, grad_b).square().item())
        W = W - step_size * grad_W
        b = b - step_size * grad_b
    f_values.append(objective(X, Y, W, b).item())
    return (
        W,
        b,
        torch.tensor(f_values, dtype=torch.float64),
        torch.tensor(grad_sq_norms, dtype=torch.float64),
    )


def accuracy(X, labels, W, b):
    return ((X @ W + b).argmax(dim=1) == labels).double().mean().item()

In [ ]:
step_size = 1.0 / L
num_steps = 500

W_T, b_T, f_values, grad_sq_norms = gradient_descent(
    X_dbl, Y_dbl, W_dbl, b_dbl, step_size, num_steps
)
k = torch.arange(1, num_steps + 1, dtype=torch.float64)

# (a) Sufficient decrease at every step.
assert torch.all(f_values[1:] <= f_values[:-1] - step_size / 2 * grad_sq_norms + 1e-12)

# (b) Objective gap against theta_ref = theta_T, for every k = 1, ..., T.
dist_sq = parameter_norm(W_dbl - W_T, b_dbl - b_T).square().item()
gap = f_values[1:] - f_values[-1]
gap_bound = dist_sq / (2 * step_size * k)
assert torch.all(gap <= gap_bound + 1e-12)

# (c) Smallest squared gradient norm seen so far.
min_grad_sq = torch.cummin(grad_sq_norms, dim=0).values
grad_bound = 2 * (f_values[0] - f_values[1:]) / (step_size * k)
assert torch.all(min_grad_sq <= grad_bound + 1e-12)

print(f"step size 1/L = {step_size:.5f}")
print(f"f(theta_0) = {f_values[0]:.4f}   f(theta_T) = {f_values[-1]:.4f}")
print(f"train accuracy = {accuracy(X_dbl, labels, W_T, b_T):.4f}")
print(f"test accuracy  = {accuracy(X_test.double(), test_labels, W_T, b_T):.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# The gap is exactly 0 at k = T, so plot k = 1, ..., T - 1 on log axes.
axes[0].loglog(k[:-1], gap[:-1], label=r"$f(\theta_k)-f(\theta_T)$")
axes[0].loglog(k[:-1], gap_bound[:-1], "--", label=r"$\|\theta_0-\theta_T\|^2/(2\eta k)$")
axes[0].set_xlabel("iteration $k$")
axes[0].set_title("(b) objective gap")
axes[0].legend()

axes[1].loglog(k, min_grad_sq, label=r"$\min_{j<k}\|\nabla f(\theta_j)\|^2$")
axes[1].loglog(k, grad_bound, "--", label=r"$2(f(\theta_0)-f(\theta_k))/(\eta k)$")
axes[1].set_xlabel("iteration $k$")
axes[1].set_title("(c) gradient norm")
axes[1].legend()

fig.tight_layout()
plt.show()